# Surveillance Test Video Downloader

This notebook downloads public surveillance/CCTV test videos organized by anomaly type for the **Conference Speech - Offline Video Analyser**.

**Categories:**
- Normal CCTV footage (streets, parking lots)
- Camera blocked / lens covered
- Camera blur / out of focus
- Camera moved / shifted
- Fire and smoke
- People detected
- Vehicles detected

**How to use:**
1. Run all cells in order
2. Download the generated zip file
3. Unzip into `ConferenceSpeech/videos/`
4. Run `python main.py` and click **Start Processing**

## Cell 1: Install Tools

In [ ]:
!pip install -q yt-dlp gdown
!which ffmpeg || echo 'ffmpeg not found - installing...' && apt-get install -y -qq ffmpeg > /dev/null 2>&1
print('Tools ready: yt-dlp, gdown, ffmpeg')

## Cell 2: Define Video Sources

Curated list of public surveillance clips by anomaly category.

**To add your own videos:** Edit the `VIDEO_SOURCES` dict below. Each entry needs a `url`, `filename`, and `max_duration` (seconds).

YouTube URLs use `yt-dlp`. Direct `.mp4` links use `wget`.

In [ ]:
import os
import json
import subprocess
import shutil
from pathlib import Path

OUTPUT_DIR = Path('/content/test_videos')
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# ============================================================
# VIDEO SOURCES - Edit/add URLs here
# ============================================================
# Supported URL types:
#   - YouTube URLs  (downloaded via yt-dlp)
#   - Direct .mp4   (downloaded via wget)
#
# max_duration: trim video to this many seconds (keeps files small)
# ============================================================

VIDEO_SOURCES = {
    'normal': [
        {
            'url': 'https://www.youtube.com/watch?v=MNn9qKG2UFI',
            'filename': 'normal_street_01.mp4',
            'max_duration': 45,
            'description': 'Normal street surveillance footage',
        },
        {
            'url': 'https://www.youtube.com/watch?v=wqctLW0Hb_0',
            'filename': 'normal_parking_01.mp4',
            'max_duration': 45,
            'description': 'Normal parking lot CCTV',
        },
    ],
    'fire_or_smoke': [
        {
            'url': 'https://www.youtube.com/watch?v=0-EkWMRFBjg',
            'filename': 'fire_cctv_01.mp4',
            'max_duration': 45,
            'description': 'Fire caught on CCTV camera',
        },
        {
            'url': 'https://www.youtube.com/watch?v=cGHSjLOuXY0',
            'filename': 'smoke_detection_01.mp4',
            'max_duration': 45,
            'description': 'Smoke visible on surveillance camera',
        },
    ],
    'person_detected': [
        {
            'url': 'https://www.youtube.com/watch?v=Gm2x6CVFpbI',
            'filename': 'person_walking_01.mp4',
            'max_duration': 45,
            'description': 'Person walking in surveillance view',
        },
        {
            'url': 'https://www.youtube.com/watch?v=aUdKzb4LGJI',
            'filename': 'person_hallway_01.mp4',
            'max_duration': 45,
            'description': 'People in hallway CCTV',
        },
    ],
    'vehicle_detected': [
        {
            'url': 'https://www.youtube.com/watch?v=PJ5xXXcfuTc',
            'filename': 'vehicle_traffic_01.mp4',
            'max_duration': 45,
            'description': 'Vehicles on road CCTV',
        },
        {
            'url': 'https://www.youtube.com/watch?v=jjlBnrzSGjc',
            'filename': 'vehicle_parking_01.mp4',
            'max_duration': 45,
            'description': 'Vehicle in parking lot surveillance',
        },
    ],
}

total = sum(len(v) for v in VIDEO_SOURCES.values())
print(f'Defined {total} videos across {len(VIDEO_SOURCES)} categories:')
for cat, vids in VIDEO_SOURCES.items():
    print(f'  {cat}: {len(vids)} video(s)')

## Cell 3: Download Videos

Downloads each video, trims to max duration, and saves to `/content/test_videos/`.

In [ ]:
def is_youtube(url):
    return 'youtube.com' in url or 'youtu.be' in url


def download_video(url, output_path, max_duration=60):
    """Download a video and trim to max_duration seconds."""
    temp_path = str(output_path) + '.tmp.mp4'

    try:
        if is_youtube(url):
            cmd = [
                'yt-dlp',
                '--no-warnings',
                '-f', 'best[height<=720][ext=mp4]/best[height<=720]/best[ext=mp4]/best',
                '--no-playlist',
                '-o', temp_path,
                url,
            ]
            result = subprocess.run(cmd, capture_output=True, text=True, timeout=120)
            if result.returncode != 0:
                print(f'  yt-dlp error: {result.stderr[:200]}')
                return False
        else:
            cmd = ['wget', '-q', '-O', temp_path, url]
            result = subprocess.run(cmd, capture_output=True, text=True, timeout=120)
            if result.returncode != 0:
                return False

        if not os.path.exists(temp_path) or os.path.getsize(temp_path) < 1000:
            print(f'  Download produced empty/tiny file')
            return False

        # Trim to max_duration using ffmpeg
        trim_cmd = [
            'ffmpeg', '-y', '-i', temp_path,
            '-t', str(max_duration),
            '-c:v', 'libx264', '-preset', 'fast', '-crf', '23',
            '-c:a', 'aac', '-b:a', '128k',
            '-movflags', '+faststart',
            str(output_path),
        ]
        result = subprocess.run(trim_cmd, capture_output=True, text=True, timeout=180)

        # Cleanup temp
        if os.path.exists(temp_path):
            os.remove(temp_path)

        if os.path.exists(str(output_path)) and os.path.getsize(str(output_path)) > 1000:
            return True
        return False

    except Exception as e:
        print(f'  Exception: {e}')
        for p in [temp_path, str(output_path)]:
            if os.path.exists(p):
                os.remove(p)
        return False


# Download all videos
results = []
total = sum(len(v) for v in VIDEO_SOURCES.values())
idx = 0

for category, videos in VIDEO_SOURCES.items():
    print(f'\n--- Category: {category} ---')
    for vid in videos:
        idx += 1
        out_path = OUTPUT_DIR / vid['filename']
        print(f'[{idx}/{total}] Downloading: {vid["filename"]} ...')

        if out_path.exists():
            print(f'  Already exists, skipping.')
            results.append({'file': vid['filename'], 'category': category, 'status': 'exists'})
            continue

        ok = download_video(vid['url'], out_path, vid.get('max_duration', 60))
        status = 'ok' if ok else 'FAILED'
        results.append({'file': vid['filename'], 'category': category, 'status': status})
        size_mb = os.path.getsize(str(out_path)) / (1024*1024) if ok else 0
        print(f'  {"Done" if ok else "FAILED"} ({size_mb:.1f} MB)' if ok else f'  FAILED')

print(f'\n=== Download complete: {sum(1 for r in results if r["status"] in ("ok","exists"))}/{total} succeeded ===')

## Cell 4: Generate Synthetic Test Videos (blocked, blur, moved)

Camera-blocked, blur, and moved videos are hard to find publicly.
This cell generates them **synthetically** from the normal footage already downloaded.

In [ ]:
import cv2
import numpy as np

def find_source_video():
    """Find any successfully downloaded video to use as source."""
    for f in sorted(OUTPUT_DIR.glob('*.mp4')):
        if f.stat().st_size > 10000:
            return str(f)
    return None


def generate_blocked_video(source_path, output_path, duration_s=30):
    """Create a 'blocked camera' video: cover lens with solid color after a few normal seconds."""
    cap = cv2.VideoCapture(source_path)
    fps = cap.get(cv2.CAP_PROP_FPS) or 25.0
    w = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
    h = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
    fourcc = cv2.VideoWriter_fourcc(*'mp4v')
    writer = cv2.VideoWriter(output_path, fourcc, fps, (w, h))

    total_frames = int(duration_s * fps)
    normal_frames = int(5 * fps)  # 5 seconds normal, then blocked

    # Pick a random dark color for the blocked cover
    block_color = (25, 20, 15)  # near-black, like a hand or tape

    for i in range(total_frames):
        ret, frame = cap.read()
        if not ret:
            cap.set(cv2.CAP_PROP_POS_FRAMES, 0)
            ret, frame = cap.read()
            if not ret:
                break

        if i >= normal_frames:
            # Gradually cover the lens (simulate hand/tape)
            alpha = min(1.0, (i - normal_frames) / (fps * 2))  # 2s fade to blocked
            overlay = np.full_like(frame, block_color, dtype=np.uint8)
            # Add slight noise to make it realistic
            noise = np.random.randint(-5, 6, frame.shape, dtype=np.int16)
            overlay = np.clip(overlay.astype(np.int16) + noise, 0, 255).astype(np.uint8)
            frame = cv2.addWeighted(frame, 1.0 - alpha, overlay, alpha, 0)

        writer.write(frame)

    cap.release()
    writer.release()
    print(f'  Created blocked video: {output_path}')


def generate_blur_video(source_path, output_path, duration_s=30):
    """Create a 'blur camera' video: camera goes out of focus after a few normal seconds."""
    cap = cv2.VideoCapture(source_path)
    fps = cap.get(cv2.CAP_PROP_FPS) or 25.0
    w = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
    h = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
    fourcc = cv2.VideoWriter_fourcc(*'mp4v')
    writer = cv2.VideoWriter(output_path, fourcc, fps, (w, h))

    total_frames = int(duration_s * fps)
    normal_frames = int(5 * fps)  # 5 seconds normal, then blur

    for i in range(total_frames):
        ret, frame = cap.read()
        if not ret:
            cap.set(cv2.CAP_PROP_POS_FRAMES, 0)
            ret, frame = cap.read()
            if not ret:
                break

        if i >= normal_frames:
            # Progressively increase blur
            progress = min(1.0, (i - normal_frames) / (fps * 3))  # 3s to max blur
            ksize = int(3 + progress * 40)  # kernel size 3 -> 43
            if ksize % 2 == 0:
                ksize += 1
            frame = cv2.GaussianBlur(frame, (ksize, ksize), 0)

        writer.write(frame)

    cap.release()
    writer.release()
    print(f'  Created blur video: {output_path}')


def generate_moved_video(source_path, output_path, duration_s=30):
    """Create a 'camera moved' video: camera shifts position mid-video."""
    cap = cv2.VideoCapture(source_path)
    fps = cap.get(cv2.CAP_PROP_FPS) or 25.0
    w = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
    h = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
    fourcc = cv2.VideoWriter_fourcc(*'mp4v')
    writer = cv2.VideoWriter(output_path, fourcc, fps, (w, h))

    total_frames = int(duration_s * fps)
    normal_frames = int(8 * fps)  # 8 seconds normal, then shift
    shift_frames = int(2 * fps)   # 2 seconds to complete shift

    max_shift_x = int(w * 0.15)  # 15% horizontal shift
    max_shift_y = int(h * 0.10)  # 10% vertical shift

    for i in range(total_frames):
        ret, frame = cap.read()
        if not ret:
            cap.set(cv2.CAP_PROP_POS_FRAMES, 0)
            ret, frame = cap.read()
            if not ret:
                break

        if i >= normal_frames:
            progress = min(1.0, (i - normal_frames) / shift_frames)
            dx = int(max_shift_x * progress)
            dy = int(max_shift_y * progress)
            M = np.float32([[1, 0, dx], [0, 1, dy]])
            frame = cv2.warpAffine(frame, M, (w, h), borderMode=cv2.BORDER_REPLICATE)

        writer.write(frame)

    cap.release()
    writer.release()
    print(f'  Created moved video: {output_path}')


# Generate synthetic videos
source = find_source_video()
if source:
    print(f'Using source video: {os.path.basename(source)}')
    print()

    print('--- Generating: camera_blocked ---')
    generate_blocked_video(source, str(OUTPUT_DIR / 'blocked_lens_01.mp4'), 30)
    generate_blocked_video(source, str(OUTPUT_DIR / 'blocked_tape_02.mp4'), 25)

    print('--- Generating: blur_frame ---')
    generate_blur_video(source, str(OUTPUT_DIR / 'blur_defocus_01.mp4'), 30)
    generate_blur_video(source, str(OUTPUT_DIR / 'blur_soft_02.mp4'), 25)

    print('--- Generating: camera_moved ---')
    generate_moved_video(source, str(OUTPUT_DIR / 'moved_shift_01.mp4'), 30)
    generate_moved_video(source, str(OUTPUT_DIR / 'moved_pan_02.mp4'), 25)

    print('\nSynthetic videos generated.')
else:
    print('ERROR: No source video found. Run Cell 3 first to download normal footage.')

## Cell 5: Organize and Preview

Lists all downloaded/generated videos with metadata and shows thumbnail previews.

In [ ]:
import cv2
import json
import os
from pathlib import Path
from IPython.display import display, HTML
import base64

manifest = []
thumbnails_html = '<div style="display:flex;flex-wrap:wrap;gap:10px;">'

for f in sorted(OUTPUT_DIR.glob('*.mp4')):
    size_mb = f.stat().st_size / (1024 * 1024)

    # Get duration and resolution
    cap = cv2.VideoCapture(str(f))
    fps = cap.get(cv2.CAP_PROP_FPS) or 25.0
    total = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    w = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
    h = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
    duration_s = total / fps if fps > 0 else 0

    # Determine category from filename
    fname = f.name.lower()
    if 'blocked' in fname or 'tape' in fname:
        cat = 'camera_blocked'
    elif 'blur' in fname or 'defocus' in fname or 'soft' in fname:
        cat = 'blur_frame'
    elif 'moved' in fname or 'pan' in fname or 'shift' in fname:
        cat = 'camera_moved'
    elif 'fire' in fname or 'smoke' in fname:
        cat = 'fire_or_smoke'
    elif 'person' in fname or 'people' in fname or 'hallway' in fname:
        cat = 'person_detected'
    elif 'vehicle' in fname or 'traffic' in fname or 'car' in fname:
        cat = 'vehicle_detected'
    else:
        cat = 'normal'

    manifest.append({
        'filename': f.name,
        'category': cat,
        'size_mb': round(size_mb, 1),
        'duration_s': round(duration_s, 1),
        'resolution': f'{w}x{h}',
        'fps': round(fps, 1),
    })

    # Extract first frame as thumbnail
    ret, thumb = cap.read()
    cap.release()
    if ret:
        thumb = cv2.resize(thumb, (200, 150))
        _, buf = cv2.imencode('.jpg', thumb)
        b64 = base64.b64encode(buf).decode('ascii')
        thumbnails_html += (
            f'<div style="text-align:center;border:1px solid #ccc;padding:4px;border-radius:6px;">'
            f'<img src="data:image/jpeg;base64,{b64}" style="border-radius:4px;"/><br/>'
            f'<small><b>{f.name}</b><br/>{cat}<br/>{duration_s:.0f}s | {size_mb:.1f}MB</small></div>'
        )

thumbnails_html += '</div>'

# Save manifest
manifest_path = OUTPUT_DIR / 'manifest.json'
with open(str(manifest_path), 'w') as mf:
    json.dump(manifest, mf, indent=2)

# Print summary table
total_size = sum(m['size_mb'] for m in manifest)
print(f'Total: {len(manifest)} videos, {total_size:.1f} MB')
print(f'Manifest saved to: {manifest_path}')
print()
print(f'{"Filename":<35} {"Category":<20} {"Duration":<10} {"Size":<10} {"Resolution"}')
print('-' * 95)
for m in manifest:
    print(f'{m["filename"]:<35} {m["category"]:<20} {m["duration_s"]:>6.1f}s   {m["size_mb"]:>6.1f}MB   {m["resolution"]}')

# Show thumbnails
display(HTML('<h3>Thumbnail Preview</h3>' + thumbnails_html))

## Cell 6: Zip and Download

Creates a zip archive and downloads it to your local machine.

After downloading, **unzip the contents into `ConferenceSpeech/videos/`** and run `python main.py`.

In [ ]:
import shutil
from google.colab import files

ZIP_NAME = 'surveillance_test_videos'
zip_path = f'/content/{ZIP_NAME}'

# Remove old zip if exists
if os.path.exists(zip_path + '.zip'):
    os.remove(zip_path + '.zip')

# Create zip (without the manifest - just videos)
shutil.make_archive(zip_path, 'zip', str(OUTPUT_DIR))

zip_size = os.path.getsize(zip_path + '.zip') / (1024 * 1024)
print(f'Zip created: {ZIP_NAME}.zip ({zip_size:.1f} MB)')
print()
print('Downloading...')
print()
print('After download:')
print('  1. Unzip the file')
print('  2. Copy all .mp4 files into ConferenceSpeech/videos/')
print('  3. Run: python main.py')
print('  4. Click "Start Processing" in the GUI')

files.download(zip_path + '.zip')

## Cell 7 (Optional): Upload to Google Drive

Alternative to browser download - saves the zip to your Google Drive.

In [ ]:
# Uncomment and run this cell to save to Google Drive instead

# from google.colab import drive
# drive.mount('/content/drive')
#
# drive_dest = '/content/drive/MyDrive/surveillance_test_videos.zip'
# shutil.copy(zip_path + '.zip', drive_dest)
# print(f'Saved to Google Drive: {drive_dest}')

## Cell 8 (Optional): Search for More CCTV Videos on YouTube

Use this to find additional surveillance clips by keyword.
Copy the URLs you like into the `VIDEO_SOURCES` dict in Cell 2 and re-run.

In [ ]:
def search_youtube(query, max_results=5):
    """Search YouTube for videos matching query and return URLs."""
    cmd = [
        'yt-dlp',
        f'ytsearch{max_results}:{query}',
        '--get-id', '--get-title', '--get-duration',
        '--no-warnings',
    ]
    result = subprocess.run(cmd, capture_output=True, text=True, timeout=30)
    if result.returncode != 0:
        print(f'Search failed: {result.stderr[:200]}')
        return []

    lines = result.stdout.strip().split('\n')
    videos = []
    # yt-dlp outputs: title, id, duration (3 lines per result)
    for i in range(0, len(lines) - 2, 3):
        title = lines[i].strip()
        vid_id = lines[i + 1].strip()
        duration = lines[i + 2].strip() if i + 2 < len(lines) else '?'
        videos.append({
            'title': title,
            'url': f'https://www.youtube.com/watch?v={vid_id}',
            'duration': duration,
        })
    return videos


# --- Search for CCTV clips ---
# Change the query below to find different types of surveillance footage
SEARCH_QUERY = 'CCTV surveillance camera footage'

print(f'Searching YouTube for: "{SEARCH_QUERY}"\n')
results = search_youtube(SEARCH_QUERY, max_results=8)

if results:
    for i, v in enumerate(results, 1):
        print(f'{i}. {v["title"]}')
        print(f'   URL: {v["url"]}')
        print(f'   Duration: {v["duration"]}')
        print()
    print('Copy any URL above into VIDEO_SOURCES in Cell 2 and re-run Cell 3.')
else:
    print('No results found. Try a different query.')